In [2]:
import duckdb
import pandas as pd
import os

In [3]:
df_dist_urg = pd.read_csv("../data/data_dist_soin/raw/dist_urgence.csv",skiprows=2,sep=';')
df_dist_urg

,Code,Libellé,Distance à la structure la plus proche 2024
0,01001,L'Abergement-Clémenciat,26
1,01002,L'Abergement-de-Varey,15
2,01004,Ambérieu-en-Bugey,0
3,01005,Ambérieux-en-Dombes,18
4,01006,Ambléon,11
...,...,...,...
34914,97615,Pamandzi,7
34915,97616,Sada,23
34916,97617,Tsingoni,19
34917,97701,Saint-Barthélemy,N/A - résultat non disponible


In [6]:
df_dist_urg[df_dist_urg['Code'].str.startswith("132")]

,Code,Libellé,Distance à la structure la plus proche 2024
4402,13201,Marseille 1er Arrondissement,2
4403,13202,Marseille 2e Arrondissement,0
4404,13203,Marseille 3e Arrondissement,0
4405,13204,Marseille 4e Arrondissement,0
4406,13205,Marseille 5e Arrondissement,0
4407,13206,Marseille 6e Arrondissement,0
4408,13207,Marseille 7e Arrondissement,2
4409,13208,Marseille 8e Arrondissement,0
4410,13209,Marseille 9e Arrondissement,6
4411,13210,Marseille 10e Arrondissement,6


In [7]:
df_dist_urg[df_dist_urg['Code'].str.startswith("693")]

,Code,Libellé,Distance à la structure la plus proche 2024
27205,69381,Lyon 1er Arrondissement,2
27206,69382,Lyon 2e Arrondissement,2
27207,69383,Lyon 3e Arrondissement,0
27208,69384,Lyon 4e Arrondissement,0
27209,69385,Lyon 5e Arrondissement,4
27210,69386,Lyon 6e Arrondissement,1
27211,69387,Lyon 7e Arrondissement,0
27212,69388,Lyon 8e Arrondissement,0
27213,69389,Lyon 9e Arrondissement,0


In [3]:
df_dist_urg[df_dist_urg['Code'].str.startswith("75")]

,Code,Libellé,Distance à la structure la plus proche 2024
29217,75101,Paris 1er Arrondissement,0
29218,75102,Paris 2e Arrondissement,0
29219,75103,Paris 3e Arrondissement,0
29220,75104,Paris 4e Arrondissement,0
29221,75105,Paris 5e Arrondissement,2
29222,75106,Paris 6e Arrondissement,3
29223,75107,Paris 7e Arrondissement,3
29224,75108,Paris 8e Arrondissement,4
29225,75109,Paris 9e Arrondissement,2
29226,75110,Paris 10e Arrondissement,0


In [4]:
df_dist_urg['Code'] = df_dist_urg['Code'].apply(lambda x: '75000' if x.startswith('75') else x)

In [6]:
df_dist_urg[df_dist_urg['Code'].str.startswith("75")]

,Code,Libellé,Distance à la structure la plus proche 2024
29217,75000,Paris 1er Arrondissement,0
29218,75000,Paris 2e Arrondissement,0
29219,75000,Paris 3e Arrondissement,0
29220,75000,Paris 4e Arrondissement,0
29221,75000,Paris 5e Arrondissement,2
29222,75000,Paris 6e Arrondissement,3
29223,75000,Paris 7e Arrondissement,3
29224,75000,Paris 8e Arrondissement,4
29225,75000,Paris 9e Arrondissement,2
29226,75000,Paris 10e Arrondissement,0


In [9]:
mapping = {
    "Code": "code_insee",
    "Libellé": "nom_commune",
    "Distance à la structure la plus proche 2024": "dist_urgence_min"
}
 # conversion DuckDB → pandas
df_dist_urg = df_dist_urg.rename(columns=mapping)



In [10]:
df_dist_urg.drop(columns=['nom_commune'],inplace=True)

In [17]:
#on supprime les lignes où dist_urg_min est "'N/A - résultat non disponible'"
df_dist_urg = df_dist_urg[~df_dist_urg['dist_urgence_min'].str.contains("N/A")]    

In [19]:
df_dist_urg['dist_urgence_min'] = df_dist_urg['dist_urgence_min'].str.replace(',','.').astype(float)

AttributeError: Can only use .str accessor with string values!

In [20]:
#on groupe par code_insee en fisant la moyenne ds distance
df_dist_urg = df_dist_urg.groupby('code_insee',as_index=False).agg({'dist_urgence_min':'mean'})

In [22]:
df_dist_urg[df_dist_urg['code_insee'].str.startswith("75")]

,code_insee,dist_urgence_min
29208,75000,1.15


In [6]:
#on retraduis en objet duckdb
df_dist_urg = duckdb.from_df(df_dist_urg)

In [47]:
df_com = duckdb.read_csv("./data/processed/communes_france_2025.csv",sep=",")
df_com

┌────────────┬─────────────────────────┬─────────────────────────┬──────────┬──────────────────────┬──────────┬──────────────┬───────────┬───────────────────────────────────────────┬────────────┬────────────────┬─────────────────┐
│ code_insee │      nom_standard       │     nom_sans_accent     │ reg_code │       reg_nom        │ dep_code │   dep_nom    │ epci_code │                 epci_nom                  │ population │ superficie_km2 │ densite_hab_km2 │
│  varchar   │         varchar         │         varchar         │  int64   │       varchar        │ varchar  │   varchar    │  varchar  │                  varchar                  │   int64    │     int64      │     double      │
├────────────┼─────────────────────────┼─────────────────────────┼──────────┼──────────────────────┼──────────┼──────────────┼───────────┼───────────────────────────────────────────┼────────────┼────────────────┼─────────────────┤
│ 01001      │ L'Abergement-Clémenciat │ l-abergement-clemenciat │       84 

In [48]:
query = """
SELECT 
    DISTINCT epci_code,
    ROUND(AVG(TRY_CAST(dist_urgence_min AS DOUBLE)),2) AS dist_urgence_moyenne
FROM df_com
LEFT JOIN df_dist_urg
ON df_com.code_insee = df_dist_urg.code_insee
GROUP BY epci_code
"""

df_dist_soin = duckdb.sql(query)

In [24]:
df_dist_pharma = pd.read_csv("../data/data_dist_soin/raw/dist_pharma.csv",skiprows=2,sep=';')
df_dist_pharma

,Code,Libellé,Distance à la pharmacie la plus proche 2024
0,01001,L'Abergement-Clémenciat,5
1,01002,L'Abergement-de-Varey,15
2,01004,Ambérieu-en-Bugey,0
3,01005,Ambérieux-en-Dombes,11
4,01006,Ambléon,11
...,...,...,...
34914,97615,Pamandzi,7
34915,97616,Sada,23
34916,97617,Tsingoni,19
34917,97701,Saint-Barthélemy,N/A - résultat non disponible


In [25]:
mapping = {
    "Code": "code_insee",
    "Libellé": "nom_commune",
    "Distance à la pharmacie la plus proche 2024": "dist_pharma_min"
}

df_dist_pharma = df_dist_pharma.rename(columns=mapping)

In [ ]:
df_dist_pharma[df_dist_pharma['code_insee'].str.startswith("75")]
df_dist_pharma['code_insee'] = df_dist_pharma['code_insee'].apply(lambda x: '75056' if x.startswith('75') else x)

,code_insee,nom_commune,dist_pharma_min
29217,75101,Paris 1er Arrondissement,0
29218,75102,Paris 2e Arrondissement,0
29219,75103,Paris 3e Arrondissement,0
29220,75104,Paris 4e Arrondissement,0
29221,75105,Paris 5e Arrondissement,0
29222,75106,Paris 6e Arrondissement,0
29223,75107,Paris 7e Arrondissement,0
29224,75108,Paris 8e Arrondissement,0
29225,75109,Paris 9e Arrondissement,0
29226,75110,Paris 10e Arrondissement,0


In [27]:
df_dist_pharma[df_dist_pharma['code_insee'].str.startswith("97")]

,code_insee,nom_commune,dist_pharma_min
34788,97101,Les Abymes,0
34789,97102,Anse-Bertrand,14
34790,97103,Baie-Mahault,0
34791,97104,Baillif,4
34792,97105,Basse-Terre,0
...,...,...,...
34914,97615,Pamandzi,7
34915,97616,Sada,23
34916,97617,Tsingoni,19
34917,97701,Saint-Barthélemy,N/A - résultat non disponible


In [51]:
query ="""
SELECT
    DISTINCT epci_code,
    ROUND(AVG(TRY_CAST(dist_pharma_min AS DOUBLE)),2) AS dist_pharma_moyenne
FROM df_com
LEFT JOIN df_dist_pharma
ON df_com.code_insee = df_dist_pharma.code_insee
GROUP BY epci_code
"""

df_dist_pharma_moy = duckdb.sql(query)

In [52]:
query = """ 
SELECT
    d.epci_code as siren,
    d.dist_urgence_moyenne,
    p.dist_pharma_moyenne
FROM df_dist_soin d
LEFT JOIN df_dist_pharma_moy p
ON d.epci_code = p.epci_code
"""

df_dist_soin_final = duckdb.sql(query)
df_dist_soin_final

┌───────────┬──────────────────────┬─────────────────────┐
│   siren   │ dist_urgence_moyenne │ dist_pharma_moyenne │
│  varchar  │        double        │       double        │
├───────────┼──────────────────────┼─────────────────────┤
│ 200071751 │                17.01 │               12.64 │
│ 240200501 │                19.34 │               12.16 │
│ 200068765 │                28.82 │               18.85 │
│ 200041465 │                46.24 │               16.34 │
│ 200040491 │                12.71 │                7.47 │
│ 240700815 │                18.38 │                9.77 │
│ 241000405 │                43.56 │                8.96 │
│ 200071777 │                35.67 │               29.31 │
│ 241200542 │                37.09 │               33.27 │
│ 200065589 │                19.83 │                7.83 │
│     ·     │                   ·  │                  ·  │
│     ·     │                   ·  │                  ·  │
│     ·     │                   ·  │                  · 

In [53]:
df_dist_soin_final.write_csv("./data/processed/distance_lieu_soin_epci.csv")